# Analytic Beam Pattern Plotting

This notebook demonstrates the beam visualization functions in `rrivis.core.jones.beam.analytic.plotting`. These let you quickly inspect beam patterns — main lobes, sidelobes, HPBW, 2D images, feed illumination — directly in a Jupyter notebook.

**Functions covered:**
1. `plot_beam_pattern()` — 1D beam cut in dB with annotations
2. `plot_beam_comparison()` — overlay multiple beam configurations
3. `plot_beam_2d()` — 2D colour-map image of beam power
4. `plot_feed_illumination()` — feed illumination across the aperture
5. `AnalyticBeamJones.plot()` — convenience method on the Jones term

In [ ]:
%matplotlib inline

import numpy as np

from rrivis.core.jones.beam.analytic.plotting import (
    plot_beam_2d,
    plot_beam_comparison,
    plot_beam_pattern,
    plot_feed_illumination,
)

## 1. Basic beam pattern (`plot_beam_pattern`)

Plot a single 1D beam cut for a 14 m dish at 150 MHz with a Gaussian taper. The plot shows the power pattern in dB, with automatic annotations for HPBW (-3 dB width), first null, and first sidelobe level.

In [ ]:
fig = plot_beam_pattern(
    diameter=14.0, frequency=150e6, taper="gaussian", edge_taper_dB=10.0
)

### Uniform taper (Airy pattern)

The uniform illumination gives the narrowest main lobe but the highest sidelobes (-17.6 dB for a circular aperture).

In [ ]:
fig = plot_beam_pattern(diameter=14.0, frequency=150e6, taper="uniform")

### Voltage overlay

Pass `show_voltage=True` to add a secondary y-axis showing the linear voltage pattern. This is useful for seeing where the voltage goes negative (phase reversal in sidelobes).

In [ ]:
fig = plot_beam_pattern(
    diameter=14.0,
    frequency=150e6,
    taper="gaussian",
    show_voltage=True,
    max_angle_deg=5.0,
)

## 2. Comparing beam configurations (`plot_beam_comparison`)

Overlay multiple beam configurations on one plot. Each entry in the `configs` list is a dict of beam parameters (any argument accepted by `compute_aperture_beam`) plus an optional `"label"` for the legend.

### Taper comparison

Compare the five available taper functions at the same diameter and frequency.

In [ ]:
fig = plot_beam_comparison(
    diameter=14.0,
    frequency=150e6,
    configs=[
        {"taper": "uniform", "label": "Uniform (Airy)"},
        {"taper": "gaussian", "label": "Gaussian 10 dB"},
        {"taper": "parabolic", "label": "Parabolic 10 dB"},
        {"taper": "parabolic_squared", "label": "Parabolic^2 10 dB"},
        {"taper": "cosine", "label": "Cosine"},
    ],
    title="Taper comparison: 14 m dish at 150 MHz",
)

### Edge taper sweep

See how increasing the edge taper (Gaussian illumination) trades off main lobe width for sidelobe suppression.

In [ ]:
fig = plot_beam_comparison(
    diameter=14.0,
    frequency=150e6,
    configs=[
        {"taper": "gaussian", "edge_taper_dB": 3.0, "label": "Gaussian 3 dB"},
        {"taper": "gaussian", "edge_taper_dB": 10.0, "label": "Gaussian 10 dB"},
        {"taper": "gaussian", "edge_taper_dB": 20.0, "label": "Gaussian 20 dB"},
        {"taper": "gaussian", "edge_taper_dB": 30.0, "label": "Gaussian 30 dB"},
    ],
    title="Edge taper sweep (Gaussian): 14 m dish at 150 MHz",
)

### Frequency comparison

The beam narrows at higher frequencies (shorter wavelength relative to dish diameter).

In [ ]:
fig = plot_beam_comparison(
    diameter=14.0,
    frequency=150e6,  # default, overridden per config
    configs=[
        {"frequency": 100e6, "label": "100 MHz"},
        {"frequency": 150e6, "label": "150 MHz"},
        {"frequency": 300e6, "label": "300 MHz"},
        {"frequency": 600e6, "label": "600 MHz"},
    ],
    max_angle_deg=15.0,
    title="Frequency comparison: 14 m Gaussian-tapered dish",
)

### Dish diameter comparison

Larger dishes produce narrower beams at the same frequency.

In [ ]:
fig = plot_beam_comparison(
    diameter=14.0,  # default, overridden per config
    frequency=150e6,
    configs=[
        {"diameter": 6.0, "label": "6 m"},
        {"diameter": 14.0, "label": "14 m"},
        {"diameter": 25.0, "label": "25 m"},
        {"diameter": 64.0, "label": "64 m"},
    ],
    max_angle_deg=15.0,
    title="Dish diameter comparison at 150 MHz (Gaussian taper)",
)

## 3. 2D beam images (`plot_beam_2d`)

A 2D colour map of beam power in dB on a Cartesian angle grid. This is especially informative for non-circular apertures where the beam shape varies with azimuth.

### Circular aperture

In [ ]:
fig = plot_beam_2d(
    diameter=14.0,
    frequency=150e6,
    aperture_shape="circular",
    taper="gaussian",
    max_angle_deg=5.0,
)

### Elliptical aperture

An elliptical aperture with different x and y diameters produces an elongated beam. The beam is narrower along the axis with the larger diameter.

In [ ]:
fig = plot_beam_2d(
    diameter=14.0,
    frequency=150e6,
    aperture_shape="elliptical",
    aperture_params={"diameter_x": 14.0, "diameter_y": 8.0},
    max_angle_deg=5.0,
    title="Elliptical aperture: Dx=14 m, Dy=8 m at 150 MHz",
)

### Rectangular aperture

In [ ]:
fig = plot_beam_2d(
    diameter=14.0,
    frequency=150e6,
    aperture_shape="rectangular",
    aperture_params={"length_x": 14.0, "length_y": 6.0},
    max_angle_deg=5.0,
    title="Rectangular aperture: 14 m x 6 m at 150 MHz",
)

## 4. Feed illumination (`plot_feed_illumination`)

Visualize how a feed pattern illuminates the dish aperture. The left panel shows the linear illumination taper from centre to edge; the right panel shows the same in dB. The dashed line marks the edge taper level.

### Corrugated horn (most common modern feed)

In [ ]:
fig = plot_feed_illumination(
    feed_model="corrugated_horn",
    feed_params={"q": 1.15},
    dish_diameter=14.0,
    focal_ratio=0.4,
)

### Open waveguide

In [ ]:
fig = plot_feed_illumination(
    feed_model="open_waveguide",
    feed_params={"b_over_lambda": 0.7},
    dish_diameter=14.0,
    focal_ratio=0.4,
)

### Dipole over ground plane

In [ ]:
fig = plot_feed_illumination(
    feed_model="dipole_ground_plane",
    feed_params={"height_wavelengths": 0.25},
    dish_diameter=14.0,
    focal_ratio=0.4,
)

### Cassegrain reflector

A Cassegrain configuration with magnification M=5 narrows the effective subtended angle at the feed, resulting in a more uniform illumination.

In [ ]:
fig = plot_feed_illumination(
    feed_model="corrugated_horn",
    feed_params={"q": 1.15},
    dish_diameter=25.0,
    focal_ratio=0.4,
    reflector_type="cassegrain",
    magnification=5.0,
    title="Cassegrain illumination: 25 m dish, M=5, corrugated horn",
)

## 5. `AnalyticBeamJones.plot()` convenience method

If you already have an `AnalyticBeamJones` instance (e.g. as part of a simulation setup), you can call `.plot()` directly without importing the plotting module.

In [ ]:
from rrivis.core.jones.beam.analytic import AnalyticBeamJones

# Create a Jones term as you would in a simulation
beam_jones = AnalyticBeamJones(
    source_altaz=np.array([[np.pi / 2, 0.0]]),  # dummy source at zenith
    frequencies=np.array([150e6, 200e6, 300e6]),
    diameter=14.0,
    taper="parabolic",
    edge_taper_dB=12.0,
)

# Plot at the default frequency (first in the array)
fig = beam_jones.plot(max_angle_deg=8.0)

In [ ]:
# Plot at a specific frequency
fig = beam_jones.plot(frequency=300e6, max_angle_deg=5.0)

## 6. Composing custom multi-panel figures

All plotting functions accept an `ax` parameter (for 1D plots), so you can compose them into custom multi-panel figures using matplotlib subplots.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

tapers = ["uniform", "gaussian", "parabolic"]
for ax, taper in zip(axes, tapers, strict=False):
    plot_beam_pattern(
        diameter=14.0,
        frequency=150e6,
        taper=taper,
        ax=ax,
        show_sidelobe_level=False,
        n_points=500,
        title=f"{taper.capitalize()} taper",
    )

fig.suptitle("Side-by-side taper comparison", fontsize=14, y=1.02)
fig.tight_layout()